# pyskills.ipynb
> Functions for view/modifying ipynb file notebook cells. Each operation returns unified diffs showing what changed. Where `exhash` is available, prefer its hash-verified editing for cell source changes.
> 
> ## Ipynb file cell editing
> 
> Cell tools take an ipynb path (expands `~`) and a cell id, e.g:
> 
>     cell_replace_lines('nb.ipynb', cell_id, 2, 3, 'replaced')
>     cell_insert_line('nb.ipynb', cell_id, 0, 'first line')
> 
> Use `summary_nb` for a one-line-per-cell overview of a large notebook, and `view_nb` to view the whole notebook (pass `only_errors=True` after running tests to jump straight to the cells that errored, with their tracebacks). Use `view_cell` to see a cell's source with line numbers before editing (`view_range` limits it to a line range). Use `find_cells` to search a notebook by regex, cell type, error state, or nbdev export directive (with grep-style `before`/`after`/`context` neighbors - `context` defaults to 1), to get a headers-only outline, or to pull out one `header_section` with its child cells. When outputs are included they are middle-truncated by default (`trunc_out`), but error outputs never are. Use `create_notebook` to start a new ipynb file, `add_cell` to insert a new cell before/after an existing cell id, and `del_cells` to delete cells.
> 
> ## Line filtering
> 
> `cell_str_replace`, `cell_strs_replace`, and `cell_del_lines` support `re_filter` and `invert_filter` for targeting only lines matching (or not matching) a regex, like ex's `g//` and `g!//`. Combine with `start_line`/`end_line` to restrict to a region.
> 
> ## Structural transforms
> 
> `python_cell(path, id, func)` rewrites a cell's source with an arbitrary `str -> str` function, and `python_cells(path, func, *ids)` sweeps every code cell (or just `ids`). `ast_cell`/`ast_cells` take a list of ast-grep `(pattern, replacement)` rules instead, e.g. `ast_cell(path, cid, [("print($X)", "log($X)")])` (requires the optional `remold` package, whose `astmap`/`cstmap` also build reusable funcs for `python_cell`).

In [ ]:
#| default_exp ipynb

In [ ]:
#| export
import difflib,re
from fastcore.utils import *
from fastcore.meta import splice_sig
from fastcore.xtras import dict2obj,truncstr
from pyskills.edit import *

In [ ]:
from tempfile import TemporaryDirectory
from fastcore.test import test_eq,expect_fail

In [ ]:
#| export
_cell_edit_doc = f"""
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages"""

In [ ]:
#| export
def _nb(fname): return Notebook.open(Path(fname).expanduser())

In [ ]:
#| export
def _cell_edit(f, name=None):
    def wrapper(fname:str, id:str|list[str], *args, update_output:bool=False, **kw):
        nb = _nb(fname)
        def _one(cid):
            cell = nb[cid]
            text = str(cell.outputs) if update_output else cell.source
            if not text: return f"error: Cell has no {'output' if update_output else 'source'}"
            try: new_text = f(text, *args, **kw)
            except Exception as e: return f'error: {e}'
            if update_output: cell.outputs = ast.literal_eval(new_text)
            else: nb[cid] = new_text
            diff = '\n'.join(list(difflib.unified_diff(text.splitlines(), new_text.splitlines(), n=1, lineterm=''))[2:])
            return diff or 'none: No changes.'
        if isinstance(id, list) or id == 'all':
            if id == 'all': id = [c.id for c in nb.cells]
            res = [(cid, r) for cid in id if not (r := _one(cid)).startswith(('error:', 'none:'))]
        else: res = PrettyString(_one(id))
        nb.save()
        return res
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _cell_edit_doc
    return res

In [ ]:
#| export
from fastcore.nbio import *

In [ ]:
tmp = TemporaryDirectory()
test_content = 'alpha\nbeta\ngamma\ndelta\n'

nb_path = f'{tmp.name}/test.ipynb'
tnb = Notebook(new_nb([test_content, 'other cell']))
tnb.save(nb_path)
cid, oid = tnb[0].id, tnb[1].id
def nb_src(): return Notebook.open(nb_path)[cid].source
cid, oid

('0453ca80', '3c1ebdb4')

In [ ]:
#| export
cell_insert_line = _cell_edit(insert_line, 'cell_insert_line')
cell_str_replace = _cell_edit(str_replace, 'cell_str_replace')
cell_strs_replace = _cell_edit(strs_replace, 'cell_strs_replace')
cell_replace_lines = _cell_edit(replace_lines, 'cell_replace_lines')
cell_del_lines = _cell_edit(del_lines, 'cell_del_lines')

In [ ]:
res = cell_insert_line(nb_path, cid, 0, 'first')
test_eq(nb_src().splitlines()[0], 'first')
res

@@ -1 +1,2 @@
+first
 alpha

In [ ]:
res = cell_str_replace(nb_path, cid, 'beta', 'BETA')
assert 'BETA' in nb_src(), f"Expected 'BETA' in cell"
assert repr(res) == str(res)
res


@@ -2,3 +2,3 @@
 alpha
-beta
+BETA
 gamma

In [ ]:
res = cell_strs_replace(nb_path, cid, ['gamma', 'delta'], ['GAMMA', 'DELTA'])
test_eq(nb_src().splitlines()[-2:], ['GAMMA', 'DELTA'])
res

@@ -3,3 +3,3 @@
 BETA
-gamma
-delta
+GAMMA
+DELTA

In [ ]:
res = cell_replace_lines(nb_path, cid, 2, 3, 'two\nthree\n')
test_eq(nb_src().splitlines()[1:3], ['two', 'three'])
res

@@ -1,4 +1,4 @@
 first
-alpha
-BETA
+two
+three
 GAMMA

In [ ]:
res = cell_del_lines(nb_path, cid, 1, 1)
test_eq(nb_src().splitlines()[0], 'two')
res

@@ -1,2 +1 @@
-first
 two

In [ ]:
#| export
def _apply_func(
    text:str,
    func:callable, # Function taking source text, returning replacement text
):
    "Replace contents with `func(text)`"
    return func(text)

python_cell = _cell_edit(_apply_func, 'python_cell')
ast_cell = _cell_edit(ast_replace, 'ast_cell')

`python_cell` rewrites a cell's source with an arbitrary `str -> str` function, such as a transform built with `remold`'s `astmap` or `cstmap`. `ast_cell` is the shortcut for a list of ast-grep `(pattern, replacement)` rules (see `ast_replace` in `pyskills.edit`; requires the optional `remold` package).

In [ ]:
cell_replace_lines(nb_path, cid, new_content="print('a')\nx = 1  # keep")
res = python_cell(nb_path, cid, lambda s: s.replace('x = 1', 'x = 2'))
test_eq(nb_src(), "print('a')\nx = 2  # keep\n")
res

@@ -1,2 +1,2 @@
 print('a')
-x = 1  # keep
+x = 2  # keep

In [ ]:
res = ast_cell(nb_path, cid, [("print($X)", "log($X)")])
test_eq(nb_src(), "log('a')\nx = 2  # keep\n")
res

@@ -1,2 +1,2 @@
-print('a')
+log('a')
 x = 2  # keep

In [ ]:
#| export
def python_cells(
    fname:str, # ipynb to edit
    func:callable, # Function taking cell source, returning replacement source
    *ids:str # cells to transform (default: all code cells)
):
    "Apply `func` to the source of each of `ids`, returning `(id, diff)` pairs for changed cells"
    if not ids: ids = [str(c.id) for c in _nb(fname).cells if c.cell_type=='code']
    return python_cell(fname, list(ids), func)

def ast_cells(
    fname:str, # ipynb to edit
    repls:list, # (pattern, replacement) ast-grep rules; replacement is a `$VAR` template or a callable(match)->str
    *ids:str # cells to transform (default: all code cells)
):
    "Apply ast-grep `repls` to the source of each of `ids`, returning `(id, diff)` pairs for changed cells"
    return python_cells(fname, lambda t: ast_replace(t, repls), *ids)

The plural forms sweep a whole notebook: every code cell by default, or just the ids given. Changed cells come back as `(id, diff)` pairs; unchanged cells are omitted.

In [ ]:
res = ast_cells(nb_path, [("log($X)", "print($X)")])
test_eq(nb_src(), "print('a')\nx = 2  # keep\n")
test_eq([i for i,_ in res], [cid])  # only changed cells report

# cells whose transform raises are skipped, not fatal: the other cells still convert
p2 = f'{tmp.name}/mixed.ipynb'
Notebook(new_nb(['x = 1', '%%bash\necho hi'])).save(p2)
def _upper_or_boom(t):
    if '%%bash' in t: raise RuntimeError('boom')
    return t.upper()
res = python_cells(p2, _upper_or_boom)
test_eq([c.source for c in Notebook.open(p2).cells], ['X = 1', '%%bash\necho hi'])
test_eq(len(res), 1)

In [ ]:
#| export
def create_notebook(
    fname:str, # path for the new ipynb; must not already exist
    source:str='', # source for its first cell
    cell_type:str='code' # 'code', 'markdown', or 'raw'
):
    "Create a new notebook containing one cell, returning that cell's id"
    p = Path(fname).expanduser()
    if p.exists(): raise FileExistsError(fname)
    cell = mk_cell(source, cell_type)
    write_nb(new_nb([cell]), p)
    return cell.id

`create_notebook` starts a fresh ipynb file. It returns the first cell's id, which anchors the `add_cell` chain that builds out the rest of the notebook - the same pattern all the cell tools use, so a whole notebook can be authored without ever touching its JSON.

In [ ]:
new_path = Path(tmp.name)/'created.ipynb'
ncid = create_notebook(new_path, '#| default_exp foo')
test_eq(Notebook.open(new_path)[ncid].source, '#| default_exp foo')
with expect_fail(FileExistsError): create_notebook(new_path)

In [ ]:
#| export
def add_cell(
    fname:str, # ipynb to edit
    source:str, # source for the new cell
    cell_type:str='code', # 'code', 'markdown', or 'raw'
    before:str=None, # id of cell to insert before
    after:str=None # id of cell to insert after
):
    "Add a new cell before/after an existing cell (pass exactly one), returning the new cell's id"
    if (before is None)==(after is None): raise ValueError('Pass exactly one of `before` or `after`')
    nb = _nb(fname)
    idx = nb.cells.index(nb[before or after]) + (after is not None)
    cell = mk_cell(source, cell_type)
    nb.cells.insert(idx, cell)
    nb.save()
    return cell.id

`add_cell` inserts a whole new cell (rather than editing within one), placed relative to an existing cell id. It returns the new cell's id so follow-up edits can target it.

In [ ]:
nid = add_cell(nb_path, 'zeta', after=cid)
test_eq([c.id for c in Notebook.open(nb_path).cells[:2]], [cid, nid])
test_eq(Notebook.open(nb_path)[nid].source, 'zeta')

mid = add_cell(nb_path, '# note', cell_type='markdown', before=cid)
test_eq((Notebook.open(nb_path)[0].id, Notebook.open(nb_path)[0].cell_type), (mid, 'markdown'))

with expect_fail(ValueError): add_cell(nb_path, 'x')
with expect_fail(ValueError): add_cell(fname=nb_path, source='x', before=cid, after=nid)

In [ ]:
#| export
def del_cells(
    fname:str, # ipynb to edit
    *ids:str # ids of cells to delete
):
    "Delete cells by id"
    nb = _nb(fname)
    for i in ids: del nb[i]
    nb.save()

`del_cells` removes whole cells. A missing id raises `KeyError`, and nothing is saved in that case.

In [ ]:
del_cells(nb_path, nid, mid)
assert nid not in Notebook.open(nb_path) and mid not in Notebook.open(nb_path)
with expect_fail(KeyError): del_cells(nb_path, 'nonexistent')

## Moving cells

`copy_cells`/`cut_cells` copy one or more cells (by id) into a small paste buffer; `paste_cells` inserts the buffered cells before/after a cell id in any notebook -- including a different file or project. To move cells across notebooks: `cut_cells` from the source, then `paste_cells` into the destination.

In [ ]:
#| export
_paste_buf = []

def copy_cells(
    fname:str, # ipynb to copy from
    *ids:str # ids of cells to copy
):
    "Copy cells into the paste buffer (replacing its contents), for later `paste_cells`"
    nb = _nb(fname)
    global _paste_buf
    _paste_buf = [(nb[i].cell_type, nb[i].source) for i in ids]


`copy_cells` reads cells into the buffer without touching the source notebook. Like `del_cells`, a missing id raises `KeyError`.

In [ ]:
copy_cells(nb_path, oid)
test_eq(_paste_buf, [('code', 'other cell')])
assert oid in Notebook.open(nb_path)  # source untouched

with expect_fail(KeyError): copy_cells(nb_path, 'nonexistent')

In [ ]:
#| export
def cut_cells(
    fname:str, # ipynb to cut from
    *ids:str # ids of cells to cut
):
    "Copy cells into the paste buffer, then delete them from `fname`"
    copy_cells(fname, *ids)
    del_cells(fname, *ids)


`cut_cells` is `copy_cells` followed by `del_cells` -- same `KeyError`-on-missing-id behavior, but the source cells are gone afterwards.

In [ ]:
cut_cells(nb_path, oid)
test_eq(_paste_buf, [('code', 'other cell')])
assert oid not in Notebook.open(nb_path)

In [ ]:
#| export
def paste_cells(
    fname:str, # ipynb to paste into
    before:str=None, # id of cell to insert before
    after:str=None # id of cell to insert after
):
    "Insert the buffered cells (from `copy_cells`/`cut_cells`) before/after a cell id, returning the new ids"
    if not _paste_buf: raise ValueError('Paste buffer is empty -- use `copy_cells`/`cut_cells` first')
    if (before is None)==(after is None): raise ValueError('Pass exactly one of `before` or `after`')
    ids,anchor,is_after = [],(after if after is not None else before),(after is not None)
    for cell_type,source in _paste_buf:
        nid = add_cell(fname, source, cell_type, **({'after':anchor} if is_after else {'before':anchor}))
        ids.append(nid)
        anchor,is_after = nid,True
    return ids

`paste_cells` inserts the buffered cells before/after a cell id -- in the same notebook, a different notebook, or a different project, since `fname` is just a path. Pasting doesn't clear the buffer, so the same copy can be pasted into several places, and multiple buffered cells keep their relative order.

In [ ]:
dst_path = f'{tmp.name}/test2.ipynb'
Notebook(new_nb(['dest cell'])).save(dst_path)
did = Notebook.open(dst_path)[0].id

new_ids = paste_cells(dst_path, after=did)
test_eq(Notebook.open(dst_path)[new_ids[0]].source, 'other cell')

paste_cells(dst_path, before=did)  # buffer still holds it -- can paste again
test_eq(Notebook.open(dst_path)[0].source, 'other cell')

In [ ]:
ord_path = f'{tmp.name}/ord.ipynb'
Notebook(new_nb(['a', 'b', 'c'])).save(ord_path)
a,b,c = [c.id for c in Notebook.open(ord_path).cells]

copy_cells(ord_path, a, b)
paste_cells(ord_path, after=c)
test_eq([c.source for c in Notebook.open(ord_path).cells], ['a', 'b', 'c', 'a', 'b'])  # order preserved

with expect_fail(ValueError): paste_cells(dst_path)  # neither before nor after
with expect_fail(ValueError): paste_cells(fname=dst_path, before=did, after=new_ids[0])  # both

## Viewing

In [ ]:
#| export
def view_cell(
    fname:str, # ipynb to get info for
    id:str, # cell id to view
    nums:bool=True, # Show line numbers?
    view_range:list=None # Optional 1-indexed (start, end) line range, end=-1 for last line
):
    "Show cell source with optional line numbers"
    res = PrettyString(_nb(fname).view(id, nums=nums))
    if not view_range: return res
    s,e = view_range
    return PrettyString('\n'.join(res.splitlines()[s-1:None if e==-1 else e]))


`view_cell` displays a cell's source with line numbers, useful for checking content before editing:

In [ ]:
res = view_cell(nb_path, cid)
assert repr(res) == str(res)
res


     1 │ print('a')
     2 │ x = 2  # keep

In [ ]:
#| hide
hp = Path.home()/'.pyskills_tilde_test.ipynb'
_tid = create_notebook(hp, 'tilde expands')
try: test_eq(view_cell(f'~/{hp.name}', _tid, nums=False).strip(), 'tilde expands')
finally: hp.unlink()

`view_range` shows just part of a big cell: a 1-indexed `(start, end)` line range, with `end=-1` meaning the last line. Line numbers stay absolute, so they remain valid addresses for line edits.

In [ ]:
full = view_cell(nb_path, cid)
test_eq(view_cell(nb_path, cid, view_range=[2,-1]), '\n'.join(full.splitlines()[1:]))
view_cell(nb_path, cid, view_range=[2,2])

     2 │ x = 2  # keep

In [ ]:
#| export
def view_cells(
    fname:str, # ipynb to get info for
    *ids:str, # ids of cells to view
    nums:bool=True # Show line numbers?
):
    "Show multiple cells' sources, each preceded by a `# cell <id>` header"
    return PrettyString('\n'.join(f'# cell {i}\n{view_cell(fname, i, nums=nums)}' for i in ids))

`view_cells` shows several cells at once, each preceded by a `# cell <id>` header (the same format `lnhashview_cells` uses):

In [ ]:
res = view_cells(ord_path, a, c)
test_eq(res, f'# cell {a}\n     1 │ a\n# cell {c}\n     1 │ c')
res

# cell 676893b8
     1 │ a
# cell 8466380d
     1 │ c

In [ ]:
#| export
def _trunc_middle(s, limit, sep='\n…\n'):
    if len(s)<=limit: return s
    i = limit//2
    return s[:i] + sep + s[len(s)-(limit-i):]

def _has_err(c): return any(o.get('output_type')=='error' for o in c.get('outputs',[]))

def _prepped(c, nums:bool=False, trunc_in:bool=False, trunc_out:bool=True):
    "Copy of cell `c` with line numbering and truncation applied; error outputs are never truncated"
    c = dict2obj(dict(c))
    src = c.source
    if nums: src = '\n'.join(f'{i+1:6d} │ {l}' for i,l in enumerate(src.splitlines()))
    if trunc_in: src = _trunc_middle(src, 80)
    c.source = src
    outs = c.get('outputs')
    if outs and trunc_out:
        errs = [o for o in outs if o.get('output_type')=='error']
        rest = [o for o in outs if o.get('output_type')!='error']
        c.outputs = ([mk_stream('stdout', _trunc_middle(render_text(rest), 100))] if rest else []) + errs
    return c

In [ ]:
#| export
def view_nb(
    fname:str, # ipynb to get info for
    incl_out:bool=False, # Include cell outputs?
    only_errors:bool=False, # Show only cells with an error output (implies `incl_out`)?
    trunc_out:bool=True, # Middle-truncate non-error outputs to ~100 chars (when included)?
    trunc_in:bool=False # Middle-truncate cell sources to ~80 chars?
):
    "Show notebook source as concise xml"
    nb = _nb(fname)
    cells = nb.cells
    if only_errors: cells,incl_out = [c for c in cells if _has_err(c)],True
    if (incl_out and trunc_out) or trunc_in: cells = [_prepped(c, trunc_in=trunc_in, trunc_out=incl_out and trunc_out) for c in cells]
    return PrettyString(cells2xml(cells, path=nb.path if incl_out and not only_errors else nb.path.name, incl_out=incl_out))

In [ ]:
view_nb(nb_path)

<nb path="test.ipynb"><code id="0453ca80">print('a')
x = 2  # keep
</code></nb>

In [ ]:
err_nb = f'{tmp.name}/err.ipynb'
enb = Notebook(new_nb(['ok = 1', 'boom']))
ecid,bid = enb[0].id, enb[1].id
enb[bid]['outputs'] = [mk_error(['Traceback (most recent call last):', 'ValueError: boom'], ename='ValueError', evalue='boom')]
enb.save(err_nb)

res = view_nb(err_nb, only_errors=True)
assert bid in res and ecid not in res
assert 'ValueError' in res

With `incl_out`, outputs are middle-truncated to ~100 chars by default (`trunc_out`), so viewing a whole notebook stays cheap even when it has big outputs; pass `trunc_out=False` for everything. `trunc_in` does the same for cell sources at ~80 chars. Error outputs are never truncated, since tracebacks are usually the point of looking.

In [ ]:
long_nb = f'{tmp.name}/long.ipynb'
lnb = Notebook(new_nb(['print("x"*300)']))
lnb[0]['outputs'] = [mk_stream('stdout', 'x'*300)]
lnb.save(long_nb)

res = view_nb(long_nb, incl_out=True)
assert '…' in res and 'x'*120 not in res
res = view_nb(long_nb, incl_out=True, trunc_out=False)
assert 'x'*300 in res

In [ ]:
res = view_nb(err_nb, only_errors=True)
assert 'ValueError: boom' in res  # error outputs are exempt from truncation

In [ ]:
#| export
def summary_nb(
    fname:str,      # ipynb to summarize
    maxlen:int=120, # truncate each cell's source to this
)->PrettyString:
    "One line per cell: id, type, and truncated/escaped source"
    def _l(c): return f"{c.id}:{c.cell_type[0]}:{truncstr(c.source.replace(chr(10), r'\n'), maxlen)}"
    return PrettyString('\n'.join(_l(c) for c in _nb(fname).cells))

`summary_nb` gives a skimmable overview - one truncated line per cell, like `nbrg`'s output but for every cell. Use it to survey a large notebook before drilling in with `view_cell` or `view_nb`.

In [ ]:
res = summary_nb(nb_path)
assert isinstance(res, PrettyString)
lines = res.splitlines()
test_eq(len(lines), len(Notebook.open(nb_path).cells))
test_eq(lines[0].split(':')[:2], [cid, 'c'])
res


0453ca80:c:print('a')\nx = 2  # keep\n

## Finding

`find_cells` searches a notebook and returns the matching cells in the same concise XML format as `view_nb`, so results are cell-id addressed and ready for follow-up viewing or editing. Filters compose: a regex (or plain substring with `use_regex=False`), cell type, nbdev-exported cells, cells with error outputs, or an id list. `before`/`after`/`context` include neighboring cells around each match, like grep's `-B`/`-A`/`-C` -- useful in nbdev notebooks where the explanation lives in the markdown cell next door.

In [ ]:
#| export
def _hdr_level(c):
    "Markdown heading level of cell `c`, or 0"
    if c.cell_type!='markdown': return 0
    m = re.match(r'(#{1,6})\s', c.source)
    return len(m.group(1)) if m else 0

_re_exp = re.compile(r'#\|\s*exports?\b')

def find_cells(
    fname:str, # ipynb to search
    re_pattern:str='', # Optional regex to search cell sources for (re.DOTALL+re.MULTILINE is used)
    cell_type:str=None, # Optional limit by cell type ('code', 'markdown', or 'raw')
    before:int=0, # Include additional n cells before matches
    after:int=0, # Include additional n cells after matches
    context:int=None, # Include additional n cells around matches (default 1, or 0 when `headers_only`)
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching (else plain substring)?
    only_err:bool=False, # Only return cells that have error outputs?
    only_exp:bool=False, # Only return cells with an nbdev `#| export` directive?
    ids:str|list[str]='', # Optionally filter by cell ids (comma-separated str, or list)
    limit:int=None, # Optionally limit number of matched cells
    incl_out:bool=False, # Include outputs in the result?
    nums:bool=False, # Show line numbers?
    trunc_out:bool=True, # Middle-truncate non-error outputs to ~100 chars (when included)?
    trunc_in:bool=False, # Middle-truncate cell sources to ~80 chars?
    headers_only:bool=False, # Only return markdown header cells, first line only?
    header_section:str=None, # Return the section starting with this header line (leading #s optional), plus its children
)->str: # Matching cells in concise XML format
    "Find cells in `fname` matching all the given criteria"
    nb = _nb(fname)
    cells = nb.cells
    if header_section:
        res,lvl = [],0
        for c in cells:
            h = _hdr_level(c)
            if res and 0<h<=lvl: break
            if res: res.append(c)
            elif h and c.source.split('\n',1)[0] in (header_section, '#'*h+' '+header_section): res,lvl = [c],h
    else:
        if re_pattern and not use_regex: re_pattern = re.escape(re_pattern)
        elif re_pattern:
            try: re.compile(re_pattern)
            except re.error: re_pattern = re.escape(re_pattern)
        flags = re.DOTALL|re.MULTILINE|(0 if use_case else re.IGNORECASE)
        if isinstance(ids,str): ids = ids.split(',') if ids else []
        ids = set(ids)
        def _match(c):
            if cell_type and c.cell_type!=cell_type: return False
            if only_err and not _has_err(c): return False
            if only_exp and not _re_exp.match(c.source): return False
            if headers_only and not _hdr_level(c): return False
            if ids and c.id not in ids: return False
            return not re_pattern or re.search(re_pattern, c.source, flags)
        idxs = [i for i,c in enumerate(cells) if _match(c)][:limit]
        if context is None: context = 0 if headers_only else 1
        before,after = max(before,context),max(after,context)
        if before or after: idxs = sorted({j for i in idxs for j in range(max(0,i-before), min(len(cells),i+after+1))})
        res = [cells[i] for i in idxs]
    if headers_only: res = [dict2obj(dict(c, source=c.source.split('\n',1)[0])) for c in res]
    if nums or trunc_in or (incl_out and trunc_out):
        res = [_prepped(c, nums=nums, trunc_in=trunc_in, trunc_out=incl_out and trunc_out) for c in res]
    return PrettyString(cells2xml(res, path=nb.path.name, incl_out=incl_out))

In [ ]:
find_path = f'{tmp.name}/find.ipynb'
fnb = Notebook(new_nb([
    mk_cell('# Adding', 'markdown'),
    mk_cell('Some prose about addition.', 'markdown'),
    mk_cell('#| export\ndef add(a,b): return a+b'),
    mk_cell('## Usage', 'markdown'),
    mk_cell('add(1,2)'),
    mk_cell('1/0'),
    mk_cell('## Edge cases', 'markdown'),
    mk_cell('add(0,0)')]))
h1,prose,exp,h2,usage,zdiv,h3,edge = [c.id for c in fnb.cells]
fnb[zdiv]['outputs'] = [mk_error(['ZeroDivisionError: division by zero'], ename='ZeroDivisionError', evalue='division by zero')]
fnb.save(find_path)

Matching is case-insensitive by default; `use_case=True` makes it exact. A pattern that fails to compile is treated as a plain substring.

In [ ]:
res = find_cells(find_path, 'RETURN')
assert exp in res and usage not in res
res = find_cells(find_path, 'RETURN', use_case=True)
assert exp not in res
res = find_cells(find_path, r'add\(\d')
assert usage in res and edge in res and exp not in res


`cell_type`, `only_exp` (nbdev `#| export` cells), `only_err` (cells with error outputs), and `limit` narrow things down; with no pattern they filter the whole notebook.

In [ ]:
res = find_cells(find_path, cell_type='markdown', context=0)
assert h1 in res and prose in res and exp not in res
res = find_cells(find_path, only_exp=True, context=0)
assert exp in res and usage not in res
res = find_cells(find_path, only_err=True, incl_out=True, context=0)
assert zdiv in res and 'ZeroDivisionError' in res
res = find_cells(find_path, 'add', limit=1, context=0)
assert h1 in res and exp not in res

`context` (or `before`/`after`) includes n neighboring cells around each match - positional neighbors of any cell type, regardless of filters. `context` defaults to 1 (0 when `headers_only`), so matches arrive with their adjacent cells; pass `context=0` for exact matches only. `nums` adds line numbers to sources, ready for line-based edits.

In [ ]:
test_eq(find_cells(find_path, only_err=True), find_cells(find_path, only_err=True, context=1))
res = find_cells(find_path, only_err=True, context=1)
assert usage in res and zdiv in res and h3 in res and exp not in res
res = find_cells(find_path, 'RETURN', nums=True)
assert '     2 │ def add(a,b): return a+b' in res

`headers_only` gives a table of contents: just the markdown header cells, first line only. `header_section` returns one section: the cell whose first line matches, plus its child cells up to the next header of the same or higher level. Leading #s in `header_section` are optional: `'Usage'` and `'## Usage'` both match a `## Usage` header (the explicit form is level-exact).

In [ ]:
res = find_cells(find_path, headers_only=True)
assert h1 in res and h2 in res and prose not in res and 'prose' not in res

res = find_cells(find_path, header_section='## Usage')
assert h2 in res and usage in res and zdiv in res
assert h3 not in res and exp not in res
test_eq(find_cells(find_path, header_section='Usage'), res)   # leading #s optional

Truncation options never touch outputs that aren't being shown -- so a notebook whose outputs can't render (here, nbformat's list-of-lines `text` form) still views fine without `incl_out`:

In [ ]:
listy_path = f'{tmp.name}/listy.ipynb'
lsnb = Notebook(new_nb(['print("hi")']))
lsnb[0]['outputs'] = [dict(output_type='execute_result', metadata={}, execution_count=1, data={'text/plain': ['hi\n','there\n']})]
lsnb.save(listy_path)

assert 'print' in find_cells(listy_path, 'print', trunc_in=True)
assert 'print' in view_nb(listy_path, trunc_in=True)

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()